In [ ]:
from ML_Project.constants import *

from ML_Project.utils.common import read_yaml, create_directories

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True) 
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [31]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        # Tout ce qui suit doit être décalé vers la droite
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,   
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config
        


In [25]:
import os
import urllib.request as request
import zipfile as zip
from ML_Project import logger
from ML_Project.utils.common import get_size


In [28]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(self.config.source_URL, self.config.local_data_file)
            logger.info(f"File {filename} has been downloaded with the following info: {headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zip.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Zip file has been extracted to {unzip_path}")



In [32]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-22 17:55:18,644]:yaml file: config/config.yaml loaded successfully:
[2026-06-22 17:55:18,645]:yaml file: params.yaml loaded successfully:
[2026-06-22 17:55:18,646]:yaml file: schema.yaml loaded successfully:
[2026-06-22 17:55:18,647]:created directory at: artifacts:
[2026-06-22 17:55:18,647]:created directory at: artifacts/data_ingestion:
[2026-06-22 17:55:19,410]:File artifacts/data_ingestion/data.zip has been downloaded with the following info: Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: FA9A:3E3CCA:109598:123D5A:6A395AE4
Accept-Ranges: bytes
Date: Mon, 22 Jun 2026 15:55:19 GMT
Via: 1.1 varnish
X-Served-By: cache-par-lfpg196